In [ ]:
!date

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import sys
import time
import os
import pandas as pd


#from stardist.matching import group_matching_labels
from skimage.io import imread
import tifffile

from collections import defaultdict

# from scipy.spatial import cKDTree
from mpl_toolkits.mplot3d import Axes3D

from matplotlib.colors import ListedColormap
plt.rcParams['figure.dpi']=200
plt.rcParams['savefig.dpi']=200

**necessary for making heatmaps and scatterplots**

In [ ]:

import sklearn
import anndata


# Plotting

**Helper Functions**

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

def overwrite_colors_per_group(color_groups, mask_id_to_color_idx, cmap_colors):
    for g in color_groups:
        IDs = g[0]
        color = matplotlib.colors.to_rgba(g[1])
        for ID in IDs:
            if mask_id_to_color_idx.get(ID) != None:
                cmap_colors[mask_id_to_color_idx.get(ID)] = color
    return cmap_colors
    

def plot_mask(mask,ax, unanntated_color_override = 'gray', color_groups = np.array([]), masks_to_color = np.array([]), mask_alpha = 0.8):
    # Ensure inputs are numpy arrays
    mask = np.asarray(mask)

    # Get unique mask IDs (excluding 0 = background)
    mask_ids = np.unique(mask)
    mask_ids = mask_ids[mask_ids != 0]
    
    # Create color map: first color is white for background
    num_colors = len(mask_ids)
    #cmap_colors = plt.cm.get_cmap('tab20', num_colors).colors  # tab20 is good for many categories
    cmap_colors = matplotlib.colormaps['tab20'].resampled(num_colors).colors
    cmap_colors = np.vstack([[1, 1, 1,1], cmap_colors])  # add white for background
    
    # Map each maskID to a color index
    mask_id_to_color_idx = {mid: i+1 for i, mid in enumerate(mask_ids)}  # +1 to leave 0 for background
    if len(color_groups) > 0:
        cmap_colors = overwrite_colors_per_group(color_groups, mask_id_to_color_idx, cmap_colors)
        
    #make a colored version of the mask, where the mask IDs are replaced with the colors they will be plotted as
    color_idx_mask = np.zeros_like(mask, dtype=int)
    for mid, cidx in mask_id_to_color_idx.items():
        color_idx_mask[mask == mid] = cidx

    # Plot
    if len(masks_to_color) > 0 and masks_to_color[0] == 'ALL_CELLS':
        masks_to_color = mask_ids
    
    #make array of len(mask_ids) with unannotated override color (gray) RBGA as each row
    override_rgba = matplotlib.colors.to_rgba(unanntated_color_override)
    override_colors = np.vstack( [[1, 1, 1,1],[override_rgba]*num_colors])
    
    #get indexes that should recieve a color from the colormap
    idxs =np.array(np.where(np.isin(mask_ids, masks_to_color)))+1
    
    #overwrite those rows with a color from cmap_colors
    override_colors[idxs] = cmap_colors[idxs]
    
    ax.imshow(color_idx_mask, cmap = ListedColormap(override_colors), interpolation='none', alpha = mask_alpha)
    

    ax.axis('off')
    mask_id_to_color_idx[0] = 0
    np.vstack( [cmap_colors,[0,0,0,0]]) #fully transparent final index for dots assigned to obscured cells
    return mask_id_to_color_idx,cmap_colors

def plot_dots(dots,mask_id_to_color_idx,cmap_colors, ax, color_override = '', name = '', edgecolor = 'black', dotsize = 4):
    # Plot dots
    check_dot_IDs(mask_id_to_color_idx, dots[:,2])
    colors = [cmap_colors[mask_id_to_color_idx[int(ID)]] for ID in dots[:,2]]
    if color_override:
        colors = [color_override for c in colors]
    
    ax.scatter(dots[:,0],dots[:,1], c = colors, s= dotsize,edgecolors=edgecolor, linewidths=0.1, label = name)

def check_dot_IDs(mask_id_to_color_idx, dot_IDs):
    for ID in dot_IDs:
        if not ID in mask_id_to_color_idx:
            print(f'Dot assigned to nuclear mask {ID} obscured by projection. Will be plotted as fully transparent (invisible)')
            mask_id_to_color_idx[ID] = -1
            
def project(nuclear_mask, z):
    projection = np.zeros((2048,2048))
    for layer in nuclear_mask[z][::-1]:
        projection = np.where(layer != 0, layer, projection) 
    return projection

def select_z(x,z):
    return x in z

def scrub_data(pool, dots, mask, pos):
    labels = pd.read_csv(f'POOL{pool}/output/annotation_coordinates_pos{pos}.csv')
    labeled_maskIDs = labels[~labels.name.apply(lambda x: x.isnumeric())]['maskID'].values #was isnumeric(x)
    dots = dots[dots.maskID.apply(lambda x: x in labeled_maskIDs)]
    
    mask = np.where(np.where(np.isin(mask, labeled_maskIDs), mask, 0), mask, 0)
    
    return dots, mask

def add_labels(pool, ax, pos, cells_to_label, offset = (20,1000,5),):
    df = pd.read_csv(f'POOL{pool}/output/annotation_coordinates_pos{pos}.csv')
    dx, dy,text_dx = offset
    df = df[np.isin(df['name'], cells_to_label)]
    N = len(df)
    fontsize = 8
    y_positions = np.arange(N)*(3*fontsize)-N/2*fontsize + dy
    x_positions = np.ones(N)*dx
    df = df.sort_values(by = 'y')
    i = 0
    for _, row in df.iterrows():
        ax.plot([row['x'],x_positions[i]], [row['y'], y_positions[i] - 0.5*fontsize], c = 'gray', linewidth = 0.3)
        ax.text(  x_positions[i] +text_dx,y_positions[i], str(row['name']), fontsize = fontsize) 
        i+=1

In [ ]:
def plot_worm(pool, pos, genes, z = list(np.arange(100)), color_by_gene = False, show_dots = True, assigned_dots_only = True, dotsize = 4,
              show_dapi = False, annotated_neurons_only = False, cells_to_label = [], label_offset = (1800,1000,5),
              mask_alpha = 0.8, dapi_offset = 5, override_edgecolor = '', cells_to_color =['ALL_CELLS'], 
              color_groups = np.array([]), save = '', return_ax = False):
    """
    pool: int
    pos: int
    genes: list or np array. genes you want to show dots for
    z: int or list. shows z stacks 0-99 by default.
    color_by_gene: default False. If True, color dots a different color for each gene. If False, color dots by their assigned nuclear mask.
    show_dots: default True. If False, don't show the dots.
    assigned_dots_only: default True. If True, don't show unassigned dots.
    dotsize: int.
    show_dapi: default False. If True, it will look for a file with f'pos{pos} DAPI.tif'. Adjust this as necessary.
    annotated_neurons_only: Default False. If False, show unannotated neuron masks as gray and only use colors for annotated neurons.
    cells_to_label: list of neuron names. When I wrote this function, we were labeling left and right. 
                    I'm not sure if it will have a bug now that two neurons can have the same name. lmk
    cells_to_color: list of mask IDs.
    label_offset: moves the label panel around.
    mask_alpha: transparency of nuclear masks.
    dapi_offset: default 5, should be 0 if the masks and z stacks of the dapi image are aligned.
    override_edgecolor: specify a color for a border around the masks. default '' means a white border (i.e. no visible border around the masks)
    save: file to save to. default '' will not save.
    return_ax: Default False. If true, return the matplotlib plot so you can further modify it (ie add a title or whatever)"""
    
    
    gene_color_override = ''
    unanntated_color_override = 'lightgray'
    edgecolor = 'white'
    
    if assigned_dots_only == False:
        edgecolor = 'black'
    if not isinstance(z, list):
        z = [z]
    #if color_by_gene:
    #     mask_color_override = 'lightgray'
            
    dots = pd.read_csv(f'POOL{pool}/output/assigned_dots_pos{pos}.csv')
    
    
    #mask = np.load(f'nuc_masks/pos{pos}/Masks_hyb_000_pos{pos}.npy',allow_pickle = True)
    mask = np.load(f'POOL{pool}/nuc_masks/pos{pos}/Masks_hyb_0_pos{pos}.npy',allow_pickle = True)

    temp = []
    for i in z:
        if i < len(mask) and i >= 0:
            temp.append(i)
    z = temp
    if len(z) == 0:
        raise ValueError(f'No valid z stacks in: {z}')
    
    if annotated_neurons_only:
        dots, mask = scrub_data(pool, dots,mask,pos)
    
    fig, ax = plt.subplots(figsize=(10, 10))
    
    if show_dapi:
        zstack = tifffile.imread(f'pos{pos} DAPI.tif')
        background_image = zstack[z[0] + dapi_offset]
        ax.imshow(background_image.T, cmap='gray', alpha=0.8) 
        edgecolor = 'white'
    
    if override_edgecolor != '':
        edgecolor = override_edgecolor
    mask = project(mask, z)
    mask_id_to_color_idx,cmap_colors = plot_mask(mask.T,ax, unanntated_color_override = unanntated_color_override, color_groups = color_groups,
                                                 mask_alpha = mask_alpha, masks_to_color = cells_to_color)

    if show_dots:
        for i, g in enumerate(genes):
            
            if color_by_gene:
                cmap = 'tab20'
                if len(genes) <= 10:
                    cmap = 'tab10'
                #colors = matplotlib.colormaps.get_cmap(cmap).colors
                #gene_color_override = colors[i%20]
                colors = matplotlib.colormaps[cmap].colors
                gene_color_override = colors[i % len(colors)]
            gene_only = dots[dots.gene == g]
            
            if np.sum(gene_only.z.apply(lambda x: select_z(x,z))) > 0:
                selection = gene_only[gene_only.z.apply(lambda x: select_z(x,z))][['x','y','maskID']].values
                if assigned_dots_only:
                    selection = gene_only[np.logical_and(gene_only.z.apply(lambda x: select_z(x,z)), gene_only.maskID > 0)][['x','y','maskID']].values

                if color_by_gene:
                    plot_dots(selection,mask_id_to_color_idx,cmap_colors, ax, color_override = gene_color_override, name = g, edgecolor = edgecolor, dotsize = dotsize)
                else:
                    
                    plot_dots(selection,mask_id_to_color_idx,cmap_colors, ax, edgecolor = edgecolor, dotsize = dotsize)
    
    add_labels(pool, ax, pos, cells_to_label, label_offset)
    if color_by_gene:
        ax.legend()

    plt.tight_layout()
    
    if return_ax:
        return ax
    if save != '':
        plt.savefig(f'figures/{save}')
    plt.show()
    

In [ ]:
def get_labeled_cell_names(pool, pos, z = np.arange(0,150)):
    """
    returns array of mask IDs and an array of cell names
    z can be an array or list of z stacks. By default, it is stacks 0-149"""
    df = pd.read_csv(f'POOl{pool}/output/cellxgene_hard_assignment_pos{pos}.csv')
    annotated_cells_ID = df.maskID[df.cell.apply(lambda x: not x.isnumeric())].values
    annotated_cells_names = df['cell'][df.cell.apply(lambda x: not x.isnumeric())].values
    return annotated_cells_ID, annotated_cells_names
def get_genes(pool, pos):
    """
    returns array of genes detected in that pool and position"""
    all_genes = pd.read_csv(f'POOL{pool}/output/assigned_dots_pos4.csv').gene.unique()
    return all_genes


In [ ]:
def group_colors(pool, pos):
    name2id = {}
    colors = ['steelblue', 'lightsteelblue', 'darkorange', 'lightsalmon', 'forestgreen', 'lightgreen', 'crimson', 
              'lightsalmon', 'mediumpurple', 'thistle', 'sienna', 'rosybrown',
              'orchid', 'pink', 'gray', 'silver', 'goldenrod', 'khaki', 'darkturquoise', 'lightblue']
    IDs, names = get_labeled_cell_names(pool, pos)
    IDs, names

    for n in np.unique(names):
        name2id[n] = IDs[np.where(names == n)]
    color_groups = []
    name2group_idx = {}
    i = 0
    for n, maskids in name2id.items():
        color_groups.append([maskids, colors[i%len(colors)]])
        name2group_idx[n] = i
        i+=1
    color_groups = np.array(color_groups, dtype = object)
    return name2group_idx, color_groups
